In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [3]:
df = pd.read_csv("../data/raw/creditcard.csv")
df.shape

(284807, 31)

In [7]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train fraud ratio:", y_train.mean())
print("y_test fruad ratio:", y_test.mean())

X_train: (227845, 30)
X_test: (56962, 30)
y_train fraud ratio: 0.001729245759178389
y_test fruad ratio: 0.0017204452090867595


In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [13]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:,1]

In [14]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962



In [15]:
cm = confusion_matrix(y_test, y_pred)
cm

array([[55478,  1386],
       [    8,    90]])

In [16]:
roc_auc = roc_auc_score(y_test, y_prob)
print("ROC-AUC:", roc_auc)

ROC-AUC: 0.9720834996210077


In [17]:
risk_score = np.round(y_prob * 100).astype(int)

In [19]:
def score_to_band(score: int) -> str:
    if score < 20:
        return "Low"
    elif score <= 70:
        return "Medium"
    return "High"

In [20]:
risk_band = [score_to_band(score) for score in risk_score]

In [21]:
results = X_test.copy()
results["ActualClass"] = y_test.values
results["PredictedClass"] = y_pred
results["FraudProbability"] = y_prob
results["RiskScore"] = risk_score
results["RiskBand"] = risk_band

In [22]:
results.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V25,V26,V27,V28,Amount,ActualClass,PredictedClass,FraudProbability,RiskScore,RiskBand
263020,160760.0,-0.674466,1.408105,-1.110622,-1.328366,1.388996,-1.308439,1.885879,-0.614233,0.311652,...,-0.135837,0.045102,0.533837,0.291319,23.00,0,0,0.005691,1,Low
11378,19847.0,-2.829816,-2.765149,2.537793,-1.074580,2.842559,-2.153536,-1.795519,-0.250020,3.073504,...,-0.027660,-0.910247,0.110802,-0.511938,11.85,0,0,0.067552,7,Low
147283,88326.0,-3.576495,2.318422,1.306985,3.263665,1.127818,2.865246,1.444125,-0.718922,1.874046,...,-0.296011,0.062823,0.552411,0.509764,76.07,0,0,0.000119,0,Low
219439,141734.0,2.060386,-0.015382,-1.082544,0.386019,-0.024331,-1.074935,0.207792,-0.338140,0.455091,...,-0.283675,0.203529,-0.063621,-0.060077,0.99,0,0,0.015131,2,Low
36939,38741.0,1.209965,1.384303,-1.343531,1.763636,0.662351,-2.113384,0.854039,-0.475963,-0.629658,...,0.818998,-0.330525,0.046884,0.104527,1.50,0,1,0.945554,95,High


In [23]:
results.sort_values("RiskScore", ascending=False).head(20)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V25,V26,V27,V28,Amount,ActualClass,PredictedClass,FraudProbability,RiskScore,RiskBand
13942,24735.0,-14.575410,9.802337,-18.043109,6.136942,-11.623105,-3.978362,-13.350186,9.829463,-2.893536,...,0.886971,-0.276769,1.723108,0.547535,89.99,0,1,1.000000,100,High
6717,8408.0,-1.813280,4.917851,-5.926130,5.701500,1.204393,-3.035138,-1.713402,0.561257,-3.796354,...,1.106766,0.323885,0.894767,0.569519,1.00,1,1,1.000000,100,High
214662,139767.0,0.467992,1.100118,-5.607145,2.204714,-0.578539,-0.174200,-3.454201,1.102823,-1.065016,...,0.319869,0.170636,0.851798,0.372098,120.54,1,1,1.000000,100,High
271571,164637.0,1.919683,2.538925,-4.751178,4.666124,2.890538,-1.753877,1.304330,-0.345229,-2.194451,...,0.400890,0.146848,0.012909,0.102838,0.77,0,1,0.999869,100,High
267760,162916.0,-0.400517,3.514639,-2.902389,3.550913,3.568916,-1.575009,2.282890,-0.257712,-3.110190,...,0.290754,0.103289,0.159134,0.308162,0.77,0,1,0.999875,100,High
150663,93853.0,-5.839192,7.151532,-12.816760,7.031115,-9.651272,-2.938427,-11.543207,4.843627,-3.494276,...,-0.275998,0.282435,0.104886,0.254417,316.06,1,1,1.000000,100,High
42009,40919.0,-2.740483,3.658095,-4.110636,5.340242,-2.666775,-0.092782,-4.388699,-0.280133,-2.821895,...,-0.403956,0.277895,0.830062,0.218690,112.33,1,1,1.000000,100,High
216376,140449.0,-2.055063,0.331810,-3.360002,1.077678,5.151589,-1.441581,0.744048,-0.103401,-0.927332,...,-0.056573,-0.250994,-0.039198,0.677591,1.00,0,1,0.997026,100,High
42756,41233.0,-10.645800,5.918307,-11.671043,8.807369,-7.975501,-3.586806,-13.616797,6.428169,-7.368451,...,-0.027898,0.354254,0.273329,-0.152908,0.00,1,1,1.000000,100,High
218442,141320.0,-6.352337,-2.370335,-4.875397,2.335045,-0.809555,-0.413647,-4.082308,2.239089,-1.986360,...,-0.240053,0.112972,0.910591,-0.650944,195.66,1,1,1.000000,100,High


In [24]:
results["RiskBand"].value_counts()
pd.crosstab(results["RiskBand"], results["ActualClass"])

ActualClass,0,1
RiskBand,,
High,635,89
Low,51164,6
Medium,5065,3
